# 03_run_medium_batch.ipynb

## Objetivo
Ejecutar un **batch mediano de validación** con textos reales del dataset para:

- validar estabilidad del pipeline con más datos
- observar comportamiento de los modelos con textos reales
- detectar patrones de error
- registrar resultados estructurados para análisis posterior

## Configuración experimental

- **Modelos**: `llama3`, `mistral`
- **Prompt**: `zero-shot`
- **Rulesets**: `R0`, `R2`
- **Tamaño del batch**: `20–30 textos`
- **Ejecución**: secuencial

## Salidas esperadas

- registros crudos de inferencia
- CSV tabular de resultados
- CSV con flags y análisis inicial
- tablas comparativas por modelo y ruleset

## Lógica metodológica

Este notebook es una **fase de validación intermedia** entre:

- la prueba pequeña del notebook 02
- y la evaluación completa con métricas automáticas y análisis extendido

Aquí se prioriza:

1. reproducibilidad
2. trazabilidad
3. estabilidad operativa
4. detección temprana de anomalías

Todavía **no** se realiza la evaluación final con métricas como SARI, BLEU, ROUGE o BERTScore.

In [1]:
# =========================
# Imports y path del proyecto
# =========================
import sys
import os
import re
import gc
import json
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("CWD:", Path.cwd().resolve())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIGS EXISTS:", (PROJECT_ROOT / "configs").exists())
print("SRC EXISTS:", (PROJECT_ROOT / "src").exists())
print("OUTPUTS EXISTS:", (PROJECT_ROOT / "outputs").exists())

CWD: /home/harielpadillasanchez/Documentos/TT/TT2/notebooks
PROJECT_ROOT: /home/harielpadillasanchez/Documentos/TT/TT2
CONFIGS EXISTS: True
SRC EXISTS: True
OUTPUTS EXISTS: True


In [2]:
# =========================
# Imports del proyecto
# =========================
from configs.models import MODELS, DEFAULT_GENERATION_CONFIG
from configs.prompts import ZERO_SHOT_TEMPLATE, build_few_shot_prompt
from configs.rules import RULESETS
from src.experiment.runner import ExperimentRunner
from src.utils.logging_utils import ExperimentLogger

print("Imports del proyecto cargados correctamente.")
print("MODELS:", list(MODELS.keys()))
print("RULESETS:", list(RULESETS.keys()))
print("DEFAULT_GENERATION_CONFIG:", DEFAULT_GENERATION_CONFIG)

Imports del proyecto cargados correctamente.
MODELS: ['bloomz_small', 'mistral', 'llama3']
RULESETS: ['R0', 'R1', 'R2', 'R3', 'R4']
DEFAULT_GENERATION_CONFIG: {'temperature': 0.2, 'top_p': 0.9, 'max_new_tokens': 256, 'repetition_penalty': 1.05, 'do_sample': False}


/home/harielpadillasanchez/Documentos/TT/TT2/.venv-bloom/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# =========================
# Configuración experimental
# =========================
EXPERIMENT_NAME = "exp_medium_batch_v1"
DATASET_NAME = "FEINA"
RANDOM_SEED = 42
N_TEXTS = 30

SELECTED_MODELS = ["llama3", "mistral"]
SELECTED_RULESETS = ["R0", "R2"]
SELECTED_PROMPT = "zero-shot"

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
LOGS_DIR = OUTPUTS_DIR / "logs"
DATA_DIR = PROJECT_ROOT / "data"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = OUTPUTS_DIR / f"{EXPERIMENT_NAME}_{RUN_TS}.csv"
ANALYSIS_CSV_PATH = OUTPUTS_DIR / f"{EXPERIMENT_NAME}_{RUN_TS}_analysis.csv"
BATCH_CSV_PATH = OUTPUTS_DIR / f"{EXPERIMENT_NAME}_{RUN_TS}_batch.csv"

print("EXPERIMENT_NAME:", EXPERIMENT_NAME)
print("DATASET_NAME:", DATASET_NAME)
print("N_TEXTS:", N_TEXTS)
print("SELECTED_MODELS:", SELECTED_MODELS)
print("SELECTED_RULESETS:", SELECTED_RULESETS)
print("SELECTED_PROMPT:", SELECTED_PROMPT)
print("CSV_PATH:", CSV_PATH)
print("ANALYSIS_CSV_PATH:", ANALYSIS_CSV_PATH)
print("BATCH_CSV_PATH:", BATCH_CSV_PATH)

EXPERIMENT_NAME: exp_medium_batch_v1
DATASET_NAME: FEINA
N_TEXTS: 30
SELECTED_MODELS: ['llama3', 'mistral']
SELECTED_RULESETS: ['R0', 'R2']
SELECTED_PROMPT: zero-shot
CSV_PATH: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/exp_medium_batch_v1_20260308_155905.csv
ANALYSIS_CSV_PATH: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/exp_medium_batch_v1_20260308_155905_analysis.csv
BATCH_CSV_PATH: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/exp_medium_batch_v1_20260308_155905_batch.csv


## Guardado de manifest y creación del runner

Se registra un manifest del experimento para que la corrida sea trazable y después se pueda relacionar con:

- configuración usada
- subset de datos
- modelos
- rulesets
- notas metodológicas

In [4]:
# =========================
# Logger y runner
# =========================
logger = ExperimentLogger(base_dir="outputs/logs")

manifest = {
    "experiment_id": EXPERIMENT_NAME,
    "goal": "Batch mediano de validación para simplificación textual con textos reales",
    "dataset_name": DATASET_NAME,
    "models": SELECTED_MODELS,
    "prompt_type": SELECTED_PROMPT,
    "rulesets": SELECTED_RULESETS,
    "n_texts": N_TEXTS,
    "default_generation_config": DEFAULT_GENERATION_CONFIG,
    "notes": [
        "Fase intermedia entre small batch y evaluación completa",
        "BLOOMZ descartado para fase principal",
        "Se registran anomalías para inspección posterior"
    ]
}

logger.save_manifest(EXPERIMENT_NAME, manifest)
print("Manifest guardado.")

runner = ExperimentRunner(experiment_id=EXPERIMENT_NAME)
print("Runner listo.")

Manifest guardado.
Runner listo.


## Carga del dataset

Esta sección localiza el archivo del dataset y carga su contenido.

**Nota:** si tu dataset real tiene otro nombre, solo ajusta la lista `CANDIDATE_DATASETS`.

In [5]:
# =========================
# Localización del dataset
# =========================
CANDIDATE_DATASETS = [
    DATA_DIR / "feina.csv",
    DATA_DIR / "feina.xlsx",
    DATA_DIR / "FEINA.xlsx",
    DATA_DIR / "Dataset_FEINA.xlsx",
    DATA_DIR / "Dataset_FEINA_10122023.xlsx",
]

dataset_path = None
for p in CANDIDATE_DATASETS:
    if p.exists():
        dataset_path = p
        break

if dataset_path is None:
    raise FileNotFoundError(
        "No se encontró el dataset en data/. Ajusta CANDIDATE_DATASETS con el nombre real del archivo."
    )

print("Dataset encontrado en:", dataset_path)

Dataset encontrado en: /home/harielpadillasanchez/Documentos/TT/TT2/data/Dataset_FEINA.xlsx


In [7]:
# =========================
# Carga del dataset
# =========================
if dataset_path.suffix.lower() == ".csv":
    df_raw = pd.read_csv(dataset_path)
elif dataset_path.suffix.lower() in [".xlsx", ".xls"]:
    df_raw = pd.read_excel(dataset_path)
else:
    raise ValueError(f"Formato no soportado: {dataset_path.suffix}")

print("Shape original:", df_raw.shape)
display(df_raw.head(3))
print("\nColumnas originales:")
print(df_raw.columns.tolist())

Shape original: (5313, 15)


,Unnamed: 0,idFinal,Segmento,Propuesta,idcod,atr0,atr1,atr2,atr3,atr4,atr5,atr6,atr7,atr8,lex
0,0,58_LibroBAC.pdf,"Como antes explicamos, las finanzas y los conc...","Como antes explicamos, las finanzas están sust...",Fiorella,15.0,21.0,8.0,5.0,NaN,NaN,NaN,NaN,NaN,0
1,1,60_LibroBAC.pdf,"Una vez dirimidos estos asuntos, se entra de l...","Una vez resueltos estos asuntos, se abordan al...",Fiorella,1.0,5.0,15.0,17.0,2.0,19.0,NaN,NaN,NaN,1
2,2,62_LibroBAC.pdf,"Pero aquí no termina la utilidad del libro, ya...","Pero aquí no termina la utilidad del libro, pu...",Fiorella,15.0,19.0,5.0,16.0,6.0,NaN,NaN,NaN,NaN,0



Columnas originales:
['Unnamed: 0', 'idFinal', 'Segmento', 'Propuesta', 'idcod', 'atr0', 'atr1', 'atr2', 'atr3', 'atr4', 'atr5', 'atr6', 'atr7', 'atr8', 'lex']


In [8]:
# =========================
# Normalización de columnas
# =========================
def normalize_col(col: str) -> str:
    col = str(col).strip().lower()
    col = re.sub(r"\s+", "_", col)
    col = (
        col.replace("á", "a")
           .replace("é", "e")
           .replace("í", "i")
           .replace("ó", "o")
           .replace("ú", "u")
           .replace("ñ", "n")
    )
    return col

df = df_raw.copy()
df.columns = [normalize_col(c) for c in df.columns]

print("Columnas normalizadas:")
print(df.columns.tolist())

Columnas normalizadas:
['unnamed:_0', 'idfinal', 'segmento', 'propuesta', 'idcod', 'atr0', 'atr1', 'atr2', 'atr3', 'atr4', 'atr5', 'atr6', 'atr7', 'atr8', 'lex']


In [10]:
# =========================
# Detección flexible de columnas clave
# =========================
possible_id_cols = ["idfinal", "id", "segment_id", "sentence_id"]
possible_source_cols = [
    "segmento",
    "segment",
    "texto",
    "text",
    "sentence",
    "texto_original",
    "original",
    "source",
    "input_text",
]
possible_target_cols = [
    "propuesta",
    "proposal",
    "texto_simplificado",
    "simplified",
    "target",
    "reference",
]

def first_existing(candidates, columns):
    for c in candidates:
        if c in columns:
            return c
    return None

ID_COL = first_existing(possible_id_cols, df.columns)
SOURCE_COL = first_existing(possible_source_cols, df.columns)
TARGET_COL = first_existing(possible_target_cols, df.columns)

print("Columnas disponibles:", df.columns.tolist())
print("ID_COL:", ID_COL)
print("SOURCE_COL:", SOURCE_COL)
print("TARGET_COL:", TARGET_COL)

if SOURCE_COL is None:
    raise ValueError(
        f"No se encontró la columna del texto original.\n"
        f"Columnas disponibles: {df.columns.tolist()}"
    )

Columnas disponibles: ['unnamed:_0', 'idfinal', 'segmento', 'propuesta', 'idcod', 'atr0', 'atr1', 'atr2', 'atr3', 'atr4', 'atr5', 'atr6', 'atr7', 'atr8', 'lex']
ID_COL: idfinal
SOURCE_COL: segmento
TARGET_COL: propuesta


In [11]:
# =========================
# Limpieza básica
# =========================
def clean_text(x):
    if pd.isna(x):
        return None
    x = str(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x if x else None

work_df = df.copy()

work_df["source_text"] = work_df[SOURCE_COL].apply(clean_text)
if TARGET_COL is not None:
    work_df["reference_text"] = work_df[TARGET_COL].apply(clean_text)
else:
    work_df["reference_text"] = None

if ID_COL is not None:
    work_df["sample_id"] = work_df[ID_COL].astype(str)
else:
    work_df["sample_id"] = [f"sample_{i}" for i in range(len(work_df))]

work_df = work_df.dropna(subset=["source_text"]).copy()
work_df = work_df.drop_duplicates(subset=["source_text"]).reset_index(drop=True)

print("Shape limpio:", work_df.shape)
display(work_df[["sample_id", "source_text", "reference_text"]].head(5))

Shape limpio: (5298, 18)


,sample_id,source_text,reference_text
0,58_LibroBAC.pdf,"Como antes explicamos, las finanzas y los conc...","Como antes explicamos, las finanzas están sust..."
1,60_LibroBAC.pdf,"Una vez dirimidos estos asuntos, se entra de l...","Una vez resueltos estos asuntos, se abordan al..."
2,62_LibroBAC.pdf,"Pero aquí no termina la utilidad del libro, ya...","Pero aquí no termina la utilidad del libro, pu..."
3,63_LibroBAC.pdf,"Por lo demás, el libro en sí, termina siendo ú...","Por lo demás, el libro es útil para personas p..."
4,69_LibroBAC.pdf,Servir de documento de lectura y consulta sobr...,Debe servir como documento de lectura y consul...


## Muestreo reproducible

Se toma una muestra reproducible del dataset para construir el batch mediano.

In [12]:
# =========================
# Muestreo reproducible
# =========================
if len(work_df) < N_TEXTS:
    raise ValueError(f"El dataset limpio solo tiene {len(work_df)} textos, menor a N_TEXTS={N_TEXTS}")

batch_df = (
    work_df.sample(n=N_TEXTS, random_state=RANDOM_SEED)
           .reset_index(drop=True)
)

print("Batch seleccionado:", batch_df.shape)
display(batch_df[["sample_id", "source_text"]].head(10))

batch_df.to_csv(BATCH_CSV_PATH, index=False, encoding="utf-8")
print("Batch exportado en:", BATCH_CSV_PATH)

Batch seleccionado: (30, 18)


,sample_id,source_text
0,5350_LibroBAC.pdf,Yo sé que esto es algo que probablemente prefe...
1,418_LibroUAC_Sincopyright.pdf,Lo calcula el Instituto Nacional de Estadístic...
2,16403_LibroBAC.pdf,Fijar el precio de los activos en el momento d...
3,17252_LibroBAC.pdf,Desviando su correspondencia hacia otra ubicac...
4,1007_LibroUide_Sincopyright.pdf,"Por esta razón, el financiero suele contar con..."
5,754_LibroUAC_Sincopyright.pdf,"El Código Tributario, por su parte, señala, en..."
6,2_LibroBAC.pdf,"Pero, algo más grave aún, o quizás por eso mis..."
7,3846_LibroBAC.pdf,"Artículos como los altos hornos, las prensas c..."
8,5775_LibroBAC.pdf,Desde luego en este grupo puede haber categorí...
9,17825_LibroBAC.pdf,Sistema económico: Forma de organización socia...


Batch exportado en: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/exp_medium_batch_v1_20260308_155905_batch.csv


## Construcción de la matriz experimental

Cada texto del batch se combina con:

- 2 modelos
- 2 rulesets
- 1 tipo de prompt

Esto produce una matriz `texto × modelo × ruleset × prompt_type`.

In [13]:
# =========================
# Matriz experimental
# =========================
rows = []

for _, row in batch_df.iterrows():
    for model_name in SELECTED_MODELS:
        for ruleset_name in SELECTED_RULESETS:
            rows.append({
                "sample_id": row["sample_id"],
                "source_text": row["source_text"],
                "reference_text": row.get("reference_text", None),
                "model_name": model_name,
                "ruleset_name": ruleset_name,
                "prompt_type": SELECTED_PROMPT,
            })

exp_df = pd.DataFrame(rows)

expected_runs = N_TEXTS * len(SELECTED_MODELS) * len(SELECTED_RULESETS)

print("Corridas esperadas:", expected_runs)
print("Corridas generadas:", len(exp_df))
assert len(exp_df) == expected_runs, "La matriz experimental no coincide con el tamaño esperado."

display(exp_df.head(8))

Corridas esperadas: 120
Corridas generadas: 120


,sample_id,source_text,reference_text,model_name,ruleset_name,prompt_type
0,5350_LibroBAC.pdf,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,llama3,R0,zero-shot
1,5350_LibroBAC.pdf,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,llama3,R2,zero-shot
2,5350_LibroBAC.pdf,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,mistral,R0,zero-shot
3,5350_LibroBAC.pdf,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,mistral,R2,zero-shot
4,418_LibroUAC_Sincopyright.pdf,Lo calcula el Instituto Nacional de Estadístic...,Lo calcula el Instituto Nacional de Estadístic...,llama3,R0,zero-shot
5,418_LibroUAC_Sincopyright.pdf,Lo calcula el Instituto Nacional de Estadístic...,Lo calcula el Instituto Nacional de Estadístic...,llama3,R2,zero-shot
6,418_LibroUAC_Sincopyright.pdf,Lo calcula el Instituto Nacional de Estadístic...,Lo calcula el Instituto Nacional de Estadístic...,mistral,R0,zero-shot
7,418_LibroUAC_Sincopyright.pdf,Lo calcula el Instituto Nacional de Estadístic...,Lo calcula el Instituto Nacional de Estadístic...,mistral,R2,zero-shot


## Helpers de análisis y utilidades

Aquí se definen funciones pequeñas para:

- contar palabras
- medir longitud
- detectar anomalías
- resumir resultados

In [14]:
# =========================
# Helpers
# =========================
def safe_len_text(x):
    if x is None or pd.isna(x):
        return 0
    return len(str(x))

def word_count(x):
    if x is None or pd.isna(x):
        return 0
    return len(str(x).split())

def compression_ratio(input_text, output_text):
    in_words = word_count(input_text)
    out_words = word_count(output_text)
    if in_words == 0:
        return np.nan
    return out_words / in_words

GENERIC_PATTERNS = [
    r"^ahora puedes\b",
    r"^este texto\b",
    r"^aqui tienes\b",
    r"^texto simplificado\b",
    r"^resumen\b",
    r"^simplificacion\b",
]

PROMPT_LEAK_PATTERNS = [
    r"texto original",
    r"texto simplificado",
    r"tu tarea es",
    r"simplifica el siguiente texto",
    r"devuelve solo",
    r"version simplificada",
    r"ejemplo\s*\d+",
]

ARTIFACT_PATTERNS = [
    r"<\|.*?\|>",
    r"\[/?inst\]",
    r"<<.*?>>",
    r"###\s*(ejemplo|tarea|respuesta)",
    r"\buser:\b",
    r"\bassistant:\b",
]

def has_pattern(text, patterns):
    if not text or pd.isna(text):
        return False
    text_low = str(text).lower()
    return any(re.search(p, text_low) for p in patterns)

def is_truncated(text):
    if not text or pd.isna(text):
        return False
    text = str(text).strip()
    if len(text) == 0:
        return False
    suspicious_endings = [",", ";", ":", " y", " o", " de", " que", " con", " para", " en", "("]
    return (
        text[-1] not in ".!?" and
        any(text.endswith(s) for s in suspicious_endings)
    )

def is_generic_output(text):
    if not text or pd.isna(text):
        return False
    text = str(text).strip().lower()
    if word_count(text) <= 4:
        return True
    return any(re.search(p, text) for p in GENERIC_PATTERNS)

## Ejecución principal

La corrida se hace de forma secuencial.  
Cada iteración:

1. llama a `runner.run_one(...)`
2. captura el resultado
3. guarda una fila en memoria para consolidar el DataFrame final
4. ejecuta limpieza ligera de memoria

In [15]:
# =========================
# Ejecución principal
# =========================
all_records = []

for i, row in exp_df.iterrows():
    print(f"[{i+1}/{len(exp_df)}] sample={row.sample_id} | model={row.model_name} | ruleset={row.ruleset_name}")

    started_at = time.perf_counter()

    try:
        record = runner.run_one(
            dataset_name=DATASET_NAME,
            model_key=row.model_name,
            prompt_type=row.prompt_type,
            ruleset=row.ruleset_name,
            source_text=row.source_text,
            sample_id=row.sample_id,
            generation_config=DEFAULT_GENERATION_CONFIG,
        )

        status = getattr(record, "status", "success")
        error_message = getattr(record, "error_message", None)
        generated_text = getattr(record, "generated_text", None)
        prompt_text = getattr(record, "prompt_text", None)
        inference_seconds = getattr(record, "inference_seconds", None)

        if inference_seconds is None:
            inference_seconds = round(time.perf_counter() - started_at, 4)

    except Exception as e:
        status = "error"
        error_message = repr(e)
        generated_text = None
        prompt_text = None
        inference_seconds = round(time.perf_counter() - started_at, 4)

    result_row = {
        "experiment_id": EXPERIMENT_NAME,
        "run_ts": RUN_TS,
        "dataset_name": DATASET_NAME,
        "sample_id": row.sample_id,
        "model_name": row.model_name,
        "ruleset_name": row.ruleset_name,
        "prompt_type": row.prompt_type,
        "source_text": row.source_text,
        "reference_text": row.reference_text,
        "generated_text": generated_text,
        "prompt_text": prompt_text,
        "status": status,
        "error_message": error_message,
        "inference_seconds": inference_seconds,
        "input_chars": safe_len_text(row.source_text),
        "output_chars": safe_len_text(generated_text),
        "input_words": word_count(row.source_text),
        "output_words": word_count(generated_text),
    }

    all_records.append(result_row)

    gc.collect()

[1/120] sample=5350_LibroBAC.pdf | model=llama3 | ruleset=R0
[2/120] sample=5350_LibroBAC.pdf | model=llama3 | ruleset=R2
[3/120] sample=5350_LibroBAC.pdf | model=mistral | ruleset=R0
[4/120] sample=5350_LibroBAC.pdf | model=mistral | ruleset=R2
[5/120] sample=418_LibroUAC_Sincopyright.pdf | model=llama3 | ruleset=R0
[6/120] sample=418_LibroUAC_Sincopyright.pdf | model=llama3 | ruleset=R2
[7/120] sample=418_LibroUAC_Sincopyright.pdf | model=mistral | ruleset=R0
[8/120] sample=418_LibroUAC_Sincopyright.pdf | model=mistral | ruleset=R2
[9/120] sample=16403_LibroBAC.pdf | model=llama3 | ruleset=R0
[10/120] sample=16403_LibroBAC.pdf | model=llama3 | ruleset=R2
[11/120] sample=16403_LibroBAC.pdf | model=mistral | ruleset=R0
[12/120] sample=16403_LibroBAC.pdf | model=mistral | ruleset=R2
[13/120] sample=17252_LibroBAC.pdf | model=llama3 | ruleset=R0
[14/120] sample=17252_LibroBAC.pdf | model=llama3 | ruleset=R2
[15/120] sample=17252_LibroBAC.pdf | model=mistral | ruleset=R0
[16/120] sample=1

In [16]:
# =========================
# Consolidación inicial
# =========================
results_df = pd.DataFrame(all_records)

print("Shape results_df:", results_df.shape)
display(results_df.head(5))

print("\nStatus:")
display(results_df["status"].value_counts(dropna=False))

Shape results_df: (120, 18)


,experiment_id,run_ts,dataset_name,sample_id,model_name,ruleset_name,prompt_type,source_text,reference_text,generated_text,prompt_text,status,error_message,inference_seconds,input_chars,output_chars,input_words,output_words
0,exp_medium_batch_v1,20260308_155905,FEINA,5350_LibroBAC.pdf,llama3,R0,zero-shot,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,Yo entiendo que no quieren hacer esto.,Reescribe en español el siguiente texto con le...,success,None,13.4410,62,38,11,7
1,exp_medium_batch_v1,20260308_155905,FEINA,5350_LibroBAC.pdf,llama3,R2,zero-shot,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,No quiero que hagan esto.,Reescribe en español el siguiente texto con le...,success,None,7.4728,62,25,11,5
2,exp_medium_batch_v1,20260308_155905,FEINA,5350_LibroBAC.pdf,mistral,R0,zero-shot,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,Esto quizás no les guste mucho hacer esto. (Th...,Reescribe en español el siguiente texto con le...,success,None,4.8468,62,97,11,17
3,exp_medium_batch_v1,20260308_155905,FEINA,5350_LibroBAC.pdf,mistral,R2,zero-shot,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,Esto quizás no les guste mucho hacer esto. (Th...,Reescribe en español el siguiente texto con le...,success,None,2.7466,62,97,11,17
4,exp_medium_batch_v1,20260308_155905,FEINA,418_LibroUAC_Sincopyright.pdf,llama3,R0,zero-shot,Lo calcula el Instituto Nacional de Estadístic...,Lo calcula el Instituto Nacional de Estadístic...,El Instituto Nacional de Estadísticas calcula ...,Reescribe en español el siguiente texto con le...,success,None,10.2968,219,213,33,34



Status:


status
success    120
Name: count, dtype: int64

In [17]:
# =========================
# Export inicial
# =========================
results_df.to_csv(CSV_PATH, index=False, encoding="utf-8")
print("CSV guardado en:", CSV_PATH)

CSV guardado en: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/exp_medium_batch_v1_20260308_155905.csv


## Flags de anomalías

Se reutilizan señales observadas en el batch pequeño:

- artifacts
- prompt leak
- truncation
- generic outputs
- outputs vacíos

In [18]:
# =========================
# Aplicación de flags
# =========================
analysis_df = results_df.copy()

analysis_df["flag_artifacts"] = analysis_df["generated_text"].apply(lambda x: has_pattern(x, ARTIFACT_PATTERNS))
analysis_df["flag_prompt_leak"] = analysis_df["generated_text"].apply(lambda x: has_pattern(x, PROMPT_LEAK_PATTERNS))
analysis_df["flag_truncation"] = analysis_df["generated_text"].apply(is_truncated)
analysis_df["flag_generic"] = analysis_df["generated_text"].apply(is_generic_output)
analysis_df["empty_output"] = analysis_df["generated_text"].isna() | (analysis_df["generated_text"].fillna("").str.strip() == "")
analysis_df["compression_ratio"] = analysis_df.apply(
    lambda r: compression_ratio(r["source_text"], r["generated_text"]), axis=1
)

flag_cols = [
    "flag_artifacts",
    "flag_prompt_leak",
    "flag_truncation",
    "flag_generic",
    "empty_output",
]

analysis_df["n_flags"] = analysis_df[flag_cols].sum(axis=1)

display(analysis_df.head(5))

,experiment_id,run_ts,dataset_name,sample_id,model_name,ruleset_name,prompt_type,source_text,reference_text,generated_text,...,output_chars,input_words,output_words,flag_artifacts,flag_prompt_leak,flag_truncation,flag_generic,empty_output,compression_ratio,n_flags
0,exp_medium_batch_v1,20260308_155905,FEINA,5350_LibroBAC.pdf,llama3,R0,zero-shot,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,Yo entiendo que no quieren hacer esto.,...,38,11,7,False,False,False,False,False,0.636364,0
1,exp_medium_batch_v1,20260308_155905,FEINA,5350_LibroBAC.pdf,llama3,R2,zero-shot,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,No quiero que hagan esto.,...,25,11,5,False,False,False,False,False,0.454545,0
2,exp_medium_batch_v1,20260308_155905,FEINA,5350_LibroBAC.pdf,mistral,R0,zero-shot,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,Esto quizás no les guste mucho hacer esto. (Th...,...,97,11,17,False,False,False,False,False,1.545455,0
3,exp_medium_batch_v1,20260308_155905,FEINA,5350_LibroBAC.pdf,mistral,R2,zero-shot,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,Esto quizás no les guste mucho hacer esto. (Th...,...,97,11,17,False,False,False,False,False,1.545455,0
4,exp_medium_batch_v1,20260308_155905,FEINA,418_LibroUAC_Sincopyright.pdf,llama3,R0,zero-shot,Lo calcula el Instituto Nacional de Estadístic...,Lo calcula el Instituto Nacional de Estadístic...,El Instituto Nacional de Estadísticas calcula ...,...,213,33,34,False,False,False,False,False,1.030303,0


In [19]:
# =========================
# Resumen global de flags
# =========================
flag_summary = pd.DataFrame({
    "flag": flag_cols,
    "count": [int(analysis_df[c].sum()) for c in flag_cols],
    "pct": [round(100 * analysis_df[c].mean(), 2) for c in flag_cols],
})

display(flag_summary.sort_values("count", ascending=False))

,flag,count,pct
2,flag_truncation,9,7.50
3,flag_generic,1,0.83
0,flag_artifacts,0,0.00
1,flag_prompt_leak,0,0.00
4,empty_output,0,0.00


## Análisis descriptivo inicial

Se comparan:

- tiempo de inferencia
- longitud de salida
- compresión
- anomalías
- diferencias por modelo
- diferencias por ruleset

In [20]:
# =========================
# Resumen global
# =========================
summary_global = analysis_df.groupby("status").agg(
    runs=("status", "size"),
    mean_time_sec=("inference_seconds", "mean"),
    median_time_sec=("inference_seconds", "median"),
    mean_output_words=("output_words", "mean"),
).reset_index()

display(summary_global)

,status,runs,mean_time_sec,median_time_sec,mean_output_words
0,success,120,5.926415,5.37785,23.783333


In [21]:
# =========================
# Comparación por modelo
# =========================
summary_by_model = analysis_df.groupby("model_name").agg(
    runs=("model_name", "size"),
    success_rate=("status", lambda s: (s == "success").mean()),
    mean_time_sec=("inference_seconds", "mean"),
    median_time_sec=("inference_seconds", "median"),
    mean_output_words=("output_words", "mean"),
    mean_compression_ratio=("compression_ratio", "mean"),
    artifact_rate=("flag_artifacts", "mean"),
    prompt_leak_rate=("flag_prompt_leak", "mean"),
    truncation_rate=("flag_truncation", "mean"),
    generic_rate=("flag_generic", "mean"),
).reset_index()

display(summary_by_model.sort_values("mean_time_sec"))

,model_name,runs,success_rate,mean_time_sec,median_time_sec,mean_output_words,mean_compression_ratio,artifact_rate,prompt_leak_rate,truncation_rate,generic_rate
1,mistral,60,1.0,5.463628,4.8871,28.133333,1.060227,0.0,0.0,0.116667,0.000000
0,llama3,60,1.0,6.389202,6.3178,19.433333,0.697544,0.0,0.0,0.033333,0.016667


In [22]:
# =========================
# Comparación por ruleset
# =========================
summary_by_ruleset = analysis_df.groupby("ruleset_name").agg(
    runs=("ruleset_name", "size"),
    success_rate=("status", lambda s: (s == "success").mean()),
    mean_time_sec=("inference_seconds", "mean"),
    median_time_sec=("inference_seconds", "median"),
    mean_output_words=("output_words", "mean"),
    mean_compression_ratio=("compression_ratio", "mean"),
    artifact_rate=("flag_artifacts", "mean"),
    prompt_leak_rate=("flag_prompt_leak", "mean"),
    truncation_rate=("flag_truncation", "mean"),
    generic_rate=("flag_generic", "mean"),
).reset_index()

display(summary_by_ruleset)

,ruleset_name,runs,success_rate,mean_time_sec,median_time_sec,mean_output_words,mean_compression_ratio,artifact_rate,prompt_leak_rate,truncation_rate,generic_rate
0,R0,60,1.0,7.693162,6.9886,24.083333,0.874286,0.0,0.0,0.066667,0.016667
1,R2,60,1.0,4.159668,4.0565,23.483333,0.883486,0.0,0.0,0.083333,0.000000


In [23]:
# =========================
# Comparación modelo vs ruleset
# =========================
summary_model_ruleset = analysis_df.groupby(["model_name", "ruleset_name"]).agg(
    runs=("sample_id", "size"),
    success_rate=("status", lambda s: (s == "success").mean()),
    mean_time_sec=("inference_seconds", "mean"),
    mean_output_words=("output_words", "mean"),
    mean_compression_ratio=("compression_ratio", "mean"),
    artifact_rate=("flag_artifacts", "mean"),
    prompt_leak_rate=("flag_prompt_leak", "mean"),
    truncation_rate=("flag_truncation", "mean"),
    generic_rate=("flag_generic", "mean"),
    avg_flags=("n_flags", "mean"),
).reset_index()

display(summary_model_ruleset.sort_values(["model_name", "ruleset_name"]))

,model_name,ruleset_name,runs,success_rate,mean_time_sec,mean_output_words,mean_compression_ratio,artifact_rate,prompt_leak_rate,truncation_rate,generic_rate,avg_flags
0,llama3,R0,30,1.0,8.197153,19.800000,0.702622,0.0,0.0,0.033333,0.033333,0.066667
1,llama3,R2,30,1.0,4.581250,19.066667,0.692466,0.0,0.0,0.033333,0.000000,0.033333
2,mistral,R0,30,1.0,7.189170,28.366667,1.045949,0.0,0.0,0.100000,0.000000,0.100000
3,mistral,R2,30,1.0,3.738087,27.900000,1.074505,0.0,0.0,0.133333,0.000000,0.133333


In [24]:
# =========================
# Tablas pivote útiles
# =========================
pivot_time = summary_model_ruleset.pivot(index="model_name", columns="ruleset_name", values="mean_time_sec")
pivot_flags = summary_model_ruleset.pivot(index="model_name", columns="ruleset_name", values="avg_flags")
pivot_compression = summary_model_ruleset.pivot(index="model_name", columns="ruleset_name", values="mean_compression_ratio")

print("Tiempo promedio")
display(pivot_time)

print("Promedio de flags")
display(pivot_flags)

print("Compresión promedio")
display(pivot_compression)

Tiempo promedio


ruleset_name,R0,R2
model_name,,
llama3,8.197153,4.581250
mistral,7.189170,3.738087


Promedio de flags


ruleset_name,R0,R2
model_name,,
llama3,0.066667,0.033333
mistral,0.100000,0.133333


Compresión promedio


ruleset_name,R0,R2
model_name,,
llama3,0.702622,0.692466
mistral,1.045949,1.074505


In [25]:
# =========================
# Casos problemáticos
# =========================
problem_cases = analysis_df[
    (analysis_df["n_flags"] > 0) | (analysis_df["status"] != "success")
].copy()

problem_cases = problem_cases.sort_values(by=["n_flags", "inference_seconds"], ascending=[False, False])

print("Casos problemáticos:", len(problem_cases))
display(problem_cases[
    [
        "sample_id", "model_name", "ruleset_name", "status",
        "flag_artifacts", "flag_prompt_leak", "flag_truncation",
        "flag_generic", "empty_output", "n_flags",
        "source_text", "generated_text"
    ]
].head(15))

Casos problemáticos: 10


,sample_id,model_name,ruleset_name,status,flag_artifacts,flag_prompt_leak,flag_truncation,flag_generic,empty_output,n_flags,source_text,generated_text
18,1007_LibroUide_Sincopyright.pdf,mistral,R0,success,False,False,True,False,False,1,"Por esta razón, el financiero suele contar con...",Financieros usan normalmente dos cargos para a...
16,1007_LibroUide_Sincopyright.pdf,llama3,R0,success,False,False,True,False,False,1,"Por esta razón, el financiero suele contar con...",El financiero cuenta con dos personas que lo a...
67,1103_LibroUAC_Sincopyright.pdf,mistral,R2,success,False,False,True,False,False,1,"De este modo, para saber el monto que se tendr...",Para saber cuánto tendrás en cinco meses ahorr...
90,11507_LibroBAC.pdf,mistral,R0,success,False,False,True,False,False,1,La importancia de un historial de crédito impe...,Importante haber un buen historial de crédito....
80,577_LibroUAC_Sincopyright.pdf,llama3,R0,success,False,False,False,True,False,1,en la situación inversa los precios suben.,Los precios aumentan.
94,14081_LibroBAC.pdf,mistral,R0,success,False,False,True,False,False,1,"Por todo lo anterior, este capítulo plantea el...",Este capítulo explica concepto y objetivos de ...
11,16403_LibroBAC.pdf,mistral,R2,success,False,False,True,False,False,1,Fijar el precio de los activos en el momento d...,Establecer el precio de los activos al emitarl...
17,1007_LibroUide_Sincopyright.pdf,llama3,R2,success,False,False,True,False,False,1,"Por esta razón, el financiero suele contar con...",El financiero tiene dos ayudantes que lo ayuda...
95,14081_LibroBAC.pdf,mistral,R2,success,False,False,True,False,False,1,"Por todo lo anterior, este capítulo plantea el...",Este capítulo explica concepto y objetivos de ...
19,1007_LibroUide_Sincopyright.pdf,mistral,R2,success,False,False,True,False,False,1,"Por esta razón, el financiero suele contar con...","Por este motivo, un financiero a menudo tiene ..."


In [26]:
# =========================
# Casos limpios
# =========================
good_cases = analysis_df[
    (analysis_df["status"] == "success") &
    (analysis_df["n_flags"] == 0)
].copy()

good_cases = good_cases.sort_values(by=["compression_ratio", "inference_seconds"], ascending=[True, True])

print("Casos limpios:", len(good_cases))
display(good_cases[
    ["sample_id", "model_name", "ruleset_name", "source_text", "generated_text", "compression_ratio"]
].head(15))

Casos limpios: 110


,sample_id,model_name,ruleset_name,source_text,generated_text,compression_ratio
21,754_LibroUAC_Sincopyright.pdf,llama3,R2,"El Código Tributario, por su parte, señala, en...",Los contribuyentes deben conservar sus libros ...,0.425532
20,754_LibroUAC_Sincopyright.pdf,llama3,R0,"El Código Tributario, por su parte, señala, en...",Los contribuyentes deben conservar sus libros ...,0.425532
113,3995_LibroBAC.pdf,llama3,R2,"Se demuestra entonces, con este ejemplo, que l...",La productividad es clave para crear más emple...,0.440000
1,5350_LibroBAC.pdf,llama3,R2,Yo sé que esto es algo que probablemente prefe...,No quiero que hagan esto.,0.454545
44,837_LibroUide_Sincopyright.pdf,llama3,R0,A través del desarrollo de los once capítulos ...,Este documento te enseñará a entender las fina...,0.456790
85,1316_LibroUAC_Sincopyright.pdf,llama3,R2,En cuanto a las remuneraciones de los trabajad...,Los honorarios de los trabajadores independien...,0.460000
45,837_LibroUide_Sincopyright.pdf,llama3,R2,A través del desarrollo de los once capítulos ...,Este documento tiene once capítulos que combin...,0.493827
50,284_LibroNEFE_Sincopyright.pdf,mistral,R0,Los acreedores confían en la información credi...,Credores confían en tus créditos pasados para ...,0.518519
23,754_LibroUAC_Sincopyright.pdf,mistral,R2,"El Código Tributario, por su parte, señala, en...",El Código Fiscal requiere que contribuyentes g...,0.553191
38,17825_LibroBAC.pdf,mistral,R0,Sistema económico: Forma de organización socia...,Sistema económico: Forma de solucionar problem...,0.555556


In [27]:
# =========================
# Export final
# =========================
analysis_df.to_csv(ANALYSIS_CSV_PATH, index=False, encoding="utf-8")

print("Archivo batch:", BATCH_CSV_PATH)
print("Archivo resultados:", CSV_PATH)
print("Archivo análisis:", ANALYSIS_CSV_PATH)

Archivo batch: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/exp_medium_batch_v1_20260308_155905_batch.csv
Archivo resultados: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/exp_medium_batch_v1_20260308_155905.csv
Archivo análisis: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/exp_medium_batch_v1_20260308_155905_analysis.csv


## Lectura metodológica final

Al terminar este notebook, conviene responder:

1. ¿El pipeline es estable con 20–30 textos?
2. ¿Qué modelo produce menos anomalías?
3. ¿Qué ruleset parece ayudar más?
4. ¿Las diferencias entre `R0` y `R2` ya son visibles?
5. ¿El tiempo de inferencia sigue siendo manejable para escalar?


In [28]:
print(analysis_df.columns.tolist())
display(analysis_df.head(2))

['experiment_id', 'run_ts', 'dataset_name', 'sample_id', 'model_name', 'ruleset_name', 'prompt_type', 'source_text', 'reference_text', 'generated_text', 'prompt_text', 'status', 'error_message', 'inference_seconds', 'input_chars', 'output_chars', 'input_words', 'output_words', 'flag_artifacts', 'flag_prompt_leak', 'flag_truncation', 'flag_generic', 'empty_output', 'compression_ratio', 'n_flags']


,experiment_id,run_ts,dataset_name,sample_id,model_name,ruleset_name,prompt_type,source_text,reference_text,generated_text,...,output_chars,input_words,output_words,flag_artifacts,flag_prompt_leak,flag_truncation,flag_generic,empty_output,compression_ratio,n_flags
0,exp_medium_batch_v1,20260308_155905,FEINA,5350_LibroBAC.pdf,llama3,R0,zero-shot,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,Yo entiendo que no quieren hacer esto.,...,38,11,7,False,False,False,False,False,0.636364,0
1,exp_medium_batch_v1,20260308_155905,FEINA,5350_LibroBAC.pdf,llama3,R2,zero-shot,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,No quiero que hagan esto.,...,25,11,5,False,False,False,False,False,0.454545,0
